# Step 1: Install Required Libraries

In [ ]:
!pip install tensorflowjs tensorflow numpy librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 58.8 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found exi

# Step 2: Upload Teachable Machine Audio Model Files

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload model.json, metadata.json, weights.bin

Saving metadata.json to metadata.json
Saving model.json to model.json
Saving weights.bin to weights.bin


# Step 3: Verify Uploaded Files

In [ ]:
import os
print("\ud83d\udcc2 Uploaded Files:", os.listdir())

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/zmq/eventloop/zmqstream.py", line 557, in _run_callback
    callback(*args, **kwargs)
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/iostream.py", line 120, in _handle_event
    event_f()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.11/dist-packages/jupyter_client/session.py", line 742, in send
    to_send = self.serialize(msg, ident)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/jupyter_client/session.py", line 630, in serialize
    content = self.pack(content)
              ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/jupyter_client/session.py", line 82, in <lambda>
    json_packer = lambda obj: jsonapi.dumps(obj, default=date_default,
                              ^

# Step 4: Load the Teachable Machine Audio Model

In [ ]:
import tensorflowjs as tfjs
import tensorflow as tf
import numpy as np
import json

with open("model.json", "r") as f:
    model_config = json.load(f)

model = tfjs.converters.load_keras_model("model.json")
print("\u2705 Model Loaded Successfully!")

✅ Model Loaded Successfully!


# Step 5: Load Class Labels

In [ ]:
with open("metadata.json", "r") as f:
    metadata = json.load(f)
# Replace "labels" with the actual key from your metadata.json file
class_labels = class_labels = metadata["wordLabels"] # Example: Assuming the key is "class_names"
print("\u2705 Class Labels:", class_labels)

✅ Class Labels: ['Background Noise', 'Bird', 'Dog', 'cat']


# Step 6: Upload an Audio File for Classification

In [ ]:
print("\ud83c\udfa7 Upload a .wav audio file to classify")
audio_file = files.upload()
audio_path = list(audio_file.keys())[0]

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/zmq/eventloop/zmqstream.py", line 557, in _run_callback
    callback(*args, **kwargs)
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/iostream.py", line 120, in _handle_event
    event_f()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/iostream.py", line 518, in _flush
    self.session.send(
  File "/usr/local/lib/python3.11/dist-packages/jupyter_client/session.py", line 742, in send
    to_send = self.serialize(msg, ident)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/jupyter_client/session.py", line 630, in serialize
    content = self.pack(content)
              ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/jupyter_client/session.py", line 82, in <lambda>
    json_packer = lambda obj: jsonapi.dumps(obj, default=date_default,
                              ^

Saving sound.mp3 to sound.mp3


# Step 7: Preprocess Audio File

In [ ]:
import librosa

def preprocess_audio(audio_path):
    y, sr = librosa.load(audio_path, sr=44100, mono=True)
    # Adjust n_mfcc and fix_length size to match model's expected input shape
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=43)  # Change n_mfcc to 43
    mfcc = librosa.util.fix_length(mfcc, size=232, axis=1) # Change size to 232

    mfcc = mfcc.astype(np.float32)
    mfcc = np.expand_dims(mfcc, axis=-1)  # Add channel dimension -> (43, 232, 1)
    mfcc = np.expand_dims(mfcc, axis=0)   # Add batch dimension -> (1, 43, 232, 1)
    return mfcc

# Step 8: Predict the Class

In [ ]:
def predict_audio(audio_path):
    audio_input = preprocess_audio(audio_path)
    prediction = model.predict(audio_input)
    predicted_class = np.argmax(prediction)
    confidence = np.max(prediction)
    return f"Prediction: {class_labels[predicted_class]} (Confidence: {confidence:.2f})"

# Step 9: Run Prediction

In [ ]:
print(predict_audio(audio_path))

1/1 [==============================] - 0s 23ms/step
Prediction: Background Noise (Confidence: 1.00)
